In [11]:
!pip -q install --upgrade pip
!pip -q install neuralforecast lightning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.0 MB/s eta 0:00:00


In [12]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm


In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [14]:
import os, glob, re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/LG AImers')

train_path  = './open/train/train.csv'
sample_path = './open/sample_submission.csv'
test_glob   = './open/test/TEST_*.csv'

train = pd.read_csv(train_path)
sample_submission = pd.read_csv(sample_path)

# NF 포맷: unique_id, ds, y
train['영업일자'] = pd.to_datetime(train['영업일자'])
train_nf = (train
            .rename(columns={'영업장명_메뉴명':'unique_id','영업일자':'ds','매출수량':'y'})
            [['unique_id','ds','y']]
            .sort_values(['unique_id','ds'])
            .reset_index(drop=True))

print('train_nf range:', train_nf['ds'].min(), '→', train_nf['ds'].max())

# 테스트 파일 목록
test_paths = sorted(glob.glob(test_glob))
assert len(test_paths) > 0, "TEST 파일을 찾지 못했습니다."


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_nf range: 2023-01-01 00:00:00 → 2024-06-15 00:00:00


In [15]:
def compute_cutoff_from_tests(paths):
    months=[]
    for p in paths:
        tdf=pd.read_csv(p)
        ds=pd.to_datetime(tdf['영업일자'])
        month_mode = ds.dt.to_period('M').mode()[0]   # 가장 많이 등장하는 달
        months.append(pd.Timestamp(month_mode.start_time))
    return min(months) - pd.Timedelta(days=1)  # 전날(보통 5/31)

cutoff_date = compute_cutoff_from_tests(test_paths)
train_cut = train_nf[train_nf['ds'] <= cutoff_date].copy()

last_train_date = train_cut['ds'].max()
END_DATE = pd.Timestamp('2025-05-31')
H_ALL = (END_DATE - last_train_date).days + 1  # 반드시 +1

print('cutoff_date:', cutoff_date)
print('last_train_date:', last_train_date)
print('H_ALL:', H_ALL)  # 기대: 2024-06-01~2025-05-31 포함 길이


cutoff_date: 2024-05-31 00:00:00
last_train_date: 2024-05-31 00:00:00
H_ALL: 366


In [16]:
# ===== 4) N-BEATS 모델 정의 & 학습 =====
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

INPUT_SIZE = 56
END_DATE = pd.Timestamp('2025-05-31')
H_ALL = (END_DATE - last_train_date).days + 1   # ★ +1 필수 (END_DATE 포함)

model = NBEATS(
    h=H_ALL,
    input_size=INPUT_SIZE,
    stack_types=['trend','seasonality','seasonality'],
    n_blocks=[2,2,2],
    # n_layers/mlp_units는 버전 따라 옵션. 지금 오류 없으면 생략해도 OK.
    loss=MAE(),               # 문자열이 아닌 객체
    max_steps=2000,
    random_seed=42,
)

nf = NeuralForecast(models=[model], freq='D')
print('>> Fitting N-BEATS (no exog) ...')
nf.fit(df=train_cut)          # ★ train_nf 말고 train_cut 로 학습


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


>> Fitting N-BEATS (no exog) ...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 9.1 M  | train
-------------------------------------------------------
7.9 M     Trainable params
1.2 M     Non-trainable params
9.1 M     Total params
36.559    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_steps=2000` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2000` reached.


In [17]:
fcst_all = nf.predict().rename(columns={'NBEATS':'매출예측'})
print('fcst range:', fcst_all['ds'].min(), '→', fcst_all['ds'].max())
# 기대: 2024-06-01 → 2025-05-31
# (필요시) 음수 방지
fcst_all['매출예측'] = fcst_all['매출예측'].clip(lower=0)


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

fcst range: 2024-06-01 00:00:00 → 2025-06-01 00:00:00


In [18]:
# TEST_xx → 목표 달(2024-06부터 순차)로 고정 매핑 + 해당 달에서만 7일 결측 계산
from dateutil.relativedelta import relativedelta

base_month = pd.Timestamp('2024-06-01')  # 시작 달 명시
test_paths_sorted = sorted(test_paths)

def missing_7_for_target_month(df_month: pd.DataFrame, target_month_start: pd.Timestamp):
    y, m = target_month_start.year, target_month_start.month
    ds = pd.to_datetime(df_month['영업일자'])

    # 해당 '목표 달'의 달력만 생성
    month_start = pd.Timestamp(y, m, 1)
    month_end   = month_start + pd.offsets.MonthEnd(1)
    full_month  = pd.date_range(month_start, month_end, freq='D')

    # 해당 달에 속하는 관측 날짜만 사용
    present = pd.DatetimeIndex(sorted(ds[(ds.dt.year == y) & (ds.dt.month == m)].unique()))
    missing = sorted(set(full_month) - set(present))

    if len(missing) != 7:
        print(f"[WARN] {y}-{m:02d}: missing={len(missing)} (expected 7). Fallback: first 7")
        missing = missing[:7]  # 방어적 처리

    return missing

label_to_date = {}  # 'TEST_00+1일' -> 'YYYY-MM-DD'

for i, tpath in enumerate(test_paths_sorted):
    tname  = os.path.basename(tpath)
    prefix = re.search(r'(TEST_\d+)', tname).group(1)
    tdf    = pd.read_csv(tpath)

    target_month_start = base_month + relativedelta(months=+i)  # TEST_00=6월, TEST_01=7월, ...
    miss = missing_7_for_target_month(tdf, target_month_start)

    for j, d in enumerate(miss, start=1):
        label_to_date[f'{prefix}+{j}일'] = d.strftime('%Y-%m-%d')

# 매핑 누락 체크
need = {s for s in sample_submission['영업일자'].unique()
        if isinstance(s, str) and s.startswith('TEST_')}
print('빠진 매핑 수:', len(need - set(label_to_date.keys())))


[WARN] 2024-06: missing=15 (expected 7). Fallback: first 7
[WARN] 2024-07: missing=20 (expected 7). Fallback: first 7
[WARN] 2024-08: missing=24 (expected 7). Fallback: first 7
[WARN] 2024-09: missing=28 (expected 7). Fallback: first 7
[WARN] 2024-10: missing=31 (expected 7). Fallback: first 7
[WARN] 2024-11: missing=30 (expected 7). Fallback: first 7
[WARN] 2024-12: missing=31 (expected 7). Fallback: first 7
[WARN] 2025-01: missing=31 (expected 7). Fallback: first 7
[WARN] 2025-02: missing=28 (expected 7). Fallback: first 7
[WARN] 2025-03: missing=31 (expected 7). Fallback: first 7
빠진 매핑 수: 0


In [19]:
# 1) sample 라벨을 실제 날짜로 치환
sub = sample_submission.copy()
sub['영업일자'] = sub['영업일자'].map(lambda s: label_to_date.get(s, s))
sub['영업일자'] = pd.to_datetime(sub['영업일자'], errors='coerce')

# 2) long 변환 → 예측과 병합
sub_long = sub.melt(id_vars='영업일자', var_name='unique_id', value_name='dummy') \
             .drop(columns='dummy').rename(columns={'영업일자':'ds'})

fcst_all2 = fcst_all.copy()
fcst_all2['ds'] = pd.to_datetime(fcst_all2['ds'])

merged = sub_long.merge(fcst_all2, on=['ds','unique_id'], how='left')
merged['매출예측'] = merged['매출예측'].fillna(0).clip(lower=0)

# Change values between 0 and 1 to 1
merged['매출예측'] = merged['매출예측'].apply(lambda x: 1 if 0 < x <= 1 else x)


# 3) wide로 복구 (원본 컬럼 순서 유지)
filled = merged.pivot(index='ds', columns='unique_id', values='매출예측')
filled = filled.reindex(columns=sub.columns[1:], fill_value=0)
filled = filled.reindex(index=sub['영업일자'])

submission_fixed = pd.DataFrame({'영업일자': sub['영업일자'].values})
submission_fixed[sub.columns[1:]] = filled.to_numpy()

submission_fixed.to_csv('submission_nbeats_no_exog_season.csv', index=False, encoding='utf-8-sig')
print('Saved -> submission_nbeats_no_exog_season.csv')
print('nonzero ratio:', (submission_fixed.iloc[:,1:]!=0).values.mean())
print(submission_fixed.iloc[:5, :5])

Saved -> submission_nbeats_no_exog_season.csv
nonzero ratio: 0.904441154700222
        영업일자  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  \
0 2024-06-01            3.115639              0.000000   
1 2024-06-02            1.000000              0.000000   
2 2024-06-03            1.000000              5.203700   
3 2024-06-04            1.000000              5.652225   
4 2024-06-05            1.250675             24.351639   

   느티나무 셀프BBQ_대여료 30,000원  느티나무 셀프BBQ_대여료 60,000원  
0               11.937739                8.596265  
1                3.736101                3.513659  
2                1.000000                1.000000  
3                1.192989                1.000000  
4                1.901526                1.091311  


In [ ]:
# 1) 샘플에 있는 TEST 라벨이 모두 매핑됐는지
need = {s for s in sample_submission['영업일자'].unique() if isinstance(s, str) and s.startswith('TEST_')}
have = set(label_to_date.keys())
print('빠진 매핑 수:', len(need - have), list((need - have))[:10])

# 2) 예측 범위 확인
print('fcst range:', fcst_all['ds'].min(), '→', fcst_all['ds'].max())


빠진 매핑 수: 0 []
fcst range: 2024-06-01 00:00:00 → 2025-06-01 00:00:00


In [24]:
import pandas as pd

# Load the submission file and sample submission file
submission_df = pd.read_csv('/content/drive/MyDrive/LG AImers/submission_nbeats_no_exog_season_modified.csv')
sample_submission = pd.read_csv('/content/drive/MyDrive/LG AImers/result/sample_submission.csv')

# Replace the first column of submission_df with the first column of sample_submission
submission_df['영업일자'] = sample_submission['영업일자']

# Save the modified DataFrame
submission_df.to_csv('submission_nbeats_no_exog_season_modified_dates.csv', index=False, encoding='utf-8-sig')

print('Modified submission file with updated dates saved as: submission_nbeats_no_exog_season_modified_dates.csv')
print(submission_df.head())

Modified submission file with updated dates saved as: submission_nbeats_no_exog_season_modified_dates.csv
         영업일자  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  \
0  TEST_00+1일            3.115639              1.000000   
1  TEST_00+2일            1.000000              1.000000   
2  TEST_00+3일            1.000000              5.203700   
3  TEST_00+4일            1.000000              5.652225   
4  TEST_00+5일            1.250675             24.351639   

   느티나무 셀프BBQ_대여료 30,000원  느티나무 셀프BBQ_대여료 60,000원  느티나무 셀프BBQ_대여료 90,000원  \
0               11.937739                8.596265                     1.0   
1                3.736101                3.513659                     1.0   
2                1.000000                1.000000                     1.0   
3                1.192989                1.000000                     1.0   
4                1.901526                1.091311                     1.0   

   느티나무 셀프BBQ_본삼겹 (단품,실내)  느티나무 셀프BBQ_스프라이트 (단체)  느티나무 셀프BBQ_신라면  \
0       